# PyIceberg Workshop: Write Path Deep Dive

PyIceberg is Python's native implementation for Apache Iceberg — no JVM required.

This notebook focuses on the **write path**: how PyIceberg creates tables, writes data, partitions it, handles overwrites, and exposes its metadata layers — all with full ACID guarantees.

---

## What This Notebook Covers

| # | Section | What You'll Do | Key Concept |
|---|---------|---------------|-------------|
| 1 | **Connect to Catalog** | Connect to REST Catalog | Lazy connection — no HTTP until first use |
| 2 | **Create Namespace** | Create `pyiceberg_demo` | Metadata-only — nothing written to S3 |
| 3 | **Create Table** | Define schema, create empty table | Field IDs are permanent — not column names |
| 4 | **Append** | Append 3 batches, trace the 6-stage write path | Parquet → Manifest → Manifest List → Atomic Commit |
| 4b | **Partitioned Append** | Partition by `sensor_id`, see per-partition files | Rows binned before writing — 1 file per partition value |
| 4c | **Overwrite** | Replace rows matching a filter | ADDED + DELETED manifest entries, no in-place mutation |
| 5 | **Inspect** | Walk snapshot history, manifest files, data files | Three metadata layers: snapshot → manifest → data file |

---

## How Each Example Is Structured

1. **What we're doing** — goal of the exercise
2. **Code** — the cell you run
3. **Expected output** — what you should see
4. **Sequence diagram** — what PyIceberg does internally
5. **What just happened** — bullet explanation

---

## Prerequisites

- Docker stack running: `docker-compose up -d`
- Python: `pip install "pyiceberg[pyarrow]" pyarrow`
- MinIO UI at [http://localhost:9001](http://localhost:9001) (admin / password)

In [ ]:
#!pip install -r requirements.txt

---
## ⚙️ Prerequisites & Workshop Setup

### What's Running (Docker Stack)

Start the stack before running this notebook:

```bash
docker-compose up -d
```

| Service | URL | Purpose |
|---------|-----|---------|
| REST Catalog | http://localhost:8181 | Iceberg catalog — tracks table locations |
| MinIO S3 | http://localhost:9000 | Object storage — Parquet + metadata files |
| MinIO Console | http://localhost:9001 | Browse S3 files visually in browser |
| PostgreSQL | localhost:5432 | Catalog backend — stores snapshot state |

### What We're Building

A table called **`sensor_readings`** with this schema:

| Field | Type | Example |
|-------|------|---------|
| `sensor_id` | string | `PICARRO-001` |
| `ts` | timestamp | `2024-01-01 00:00:00` |
| `co2_ppm` | double | `405.3` |
| `ch4_ppb` | double | `1872.1` |

### Workshop Flow
1. Connect to REST Catalog
2. Create namespace + table
3. Append 3 batches (50 rows each) → 3 snapshots
4. Inspect: snapshots, manifests, data files
5. Read: filters, column projection, time travel
6. Evolve schema, inspect metadata

> **Run the cell below first** to confirm all services are reachable before we start.

In [ ]:
# ─── Infrastructure Health Check ────────────────────────────────────────────
# Run this first! Confirms all Docker services are reachable before we start.

import urllib.request
import urllib.error

checks = [
    ("REST Catalog",  "http://localhost:8181/v1/config"),
    ("MinIO S3",      "http://localhost:9000/minio/health/live"),
    ("MinIO Console", "http://localhost:9001"),
]

print("Checking workshop infrastructure...\n")
all_ok = True

for name, url in checks:
    try:
        urllib.request.urlopen(url, timeout=3)
        print(f"  ✅  {name:<20s}  UP    {url}")
    except urllib.error.HTTPError as e:
        # HTTPError means the server responded — service is up
        print(f"  ✅  {name:<20s}  UP    {url}")
    except Exception as e:
        print(f"  ❌  {name:<20s}  DOWN  — {e}")
        all_ok = False

print()
if all_ok:
    print("✅  All services are up — ready to start the workshop!")
else:
    print("❌  One or more services are down.")
    print("    Run:  docker-compose up -d")
    print("    Then wait ~10 seconds and re-run this cell.")

---
## 1. Connect to REST Catalog

**What we're doing:** Creating a client that can talk to our Iceberg REST Catalog. The catalog is the central registry that knows where every table's metadata lives. Think of it as a phone book — it maps table names to S3 locations.

In [ ]:
from pyiceberg.catalog.rest import RestCatalog

catalog = RestCatalog(
    name="workshop",
    **{
        "uri": "http://localhost:8181",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.path-style-access": "true",
        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO",
    },
)
print("Connected to REST Catalog")

**Expected output:**
```
Connected to REST Catalog
```

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py

    Code->>RC: RestCatalog(name, uri, **s3_config)
    RC->>RC: __init__() stores uri + s3 config in memory
    RC-->>Code: RestCatalog object (no HTTP yet)

    Note over Code,RC: Connection is LAZY — no network call until you use the catalog
```

**What just happened:**
- `RestCatalog.__init__()` simply stores the URI and S3 credentials in memory
- No HTTP request is made — PyIceberg uses **lazy connection** (connects on first use)
- `py-io-impl` tells PyIceberg to use `PyArrowFileIO` for all file I/O (reading Parquet, Avro)
- The first real network call happens when you call `list_namespaces()` or `create_table()`

---
## 2. Create Namespace

**What we're doing:** Creating a logical grouping called `pyiceberg_demo` for our tables. Namespaces are like schemas in a database — they organize tables but don't create anything on S3.

In [ ]:
NS = ("pyiceberg_demo",)

if NS not in catalog.list_namespaces():
    catalog.create_namespace(NS)
    print(f"Namespace '{NS[0]}' created")
else:
    print(f"Namespace '{NS[0]}' already exists")

**Expected output:**
```
Namespace 'pyiceberg_demo' created
```
*(or `already exists` if re-running)*

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py
    participant SRV as REST Catalog Server<br/><br/>localhost:8181
    participant PG as PostgreSQL<br/><br/>Metadata Store

    Code->>RC: catalog.list_namespaces()
    RC->>SRV: GET /v1/namespaces
    SRV->>PG: SELECT namespaces
    PG-->>SRV: []
    SRV-->>RC: {"namespaces": []}
    RC-->>Code: [] → not found, proceed

    Code->>RC: catalog.create_namespace("pyiceberg_demo")
    RC->>SRV: POST /v1/namespaces<br/>{"namespace": ["pyiceberg_demo"]}
    SRV->>PG: INSERT namespace row
    PG-->>SRV: ✅ done
    SRV-->>RC: 200 OK
    RC-->>Code: None

    Note over Code,PG: No S3 / MinIO involved — namespace is metadata only
```

**What just happened:**
- `catalog.list_namespaces()` makes the **first HTTP call** — `GET /v1/namespaces`
- `catalog.create_namespace()` sends `POST /v1/namespaces` with `{"namespace": ["pyiceberg_demo"]}`
- The catalog server inserts one row in PostgreSQL's `iceberg_namespace` table
- **Nothing is written to MinIO/S3** — namespaces are pure metadata
- Notice `schema.py` is NOT involved here — no field IDs, no schema, just a name

---
## 3. Define Schema & Create Table

**What we're doing:** Defining a 4-field schema for a `sensor_readings` table and creating it in the catalog. The fields represent Picarro gas analyzer data: sensor ID, timestamp, CO2 (ppm), and CH4 (ppb).

In [ ]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField, StringType, DoubleType, TimestampType,
)

TABLE_ID = (*NS, "sensor_readings")

schema = Schema(
    NestedField(1, "sensor_id", StringType()),
    NestedField(2, "ts",        TimestampType()),
    NestedField(3, "co2_ppm",   DoubleType()),
    NestedField(4, "ch4_ppb",   DoubleType()),
)

# Drop if exists (clean slate for workshop)
try:
    catalog.load_table(TABLE_ID)
    catalog.drop_table(TABLE_ID)
    print("(dropped existing table for clean run)")
except Exception:
    pass

table = catalog.create_table(TABLE_ID, schema=schema)
print(f"Table '{'.'.join(TABLE_ID)}' created")
print(f"Schema: {table.schema()}")

**Expected output:**
```
(dropped existing table for clean run)      ← only if re-running
Table 'pyiceberg_demo.sensor_readings' created
Schema: table {
  1: sensor_id: optional string
  2: ts: optional timestamp
  3: co2_ppm: optional double
  4: ch4_ppb: optional double
}
```

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py
    participant SC as schema.py<br/><br/>assign_fresh_schema_ids()
    participant SRV as REST Catalog Server<br/><br/>localhost:8181
    participant S3 as MinIO / S3<br/><br/>Object Storage
    participant PG as PostgreSQL<br/><br/>Metadata Store

    Code->>RC: catalog.create_table(identifier, schema, partition_spec)

    Note over RC,SC: ── Client-side only — no network calls yet ──

    RC->>SC: assign_fresh_schema_ids(schema)
    SC->>SC: Traverse each field<br/>assign monotonically increasing IDs
    SC-->>RC: fresh_schema<br/>sensor_id→1  ts→2  co2_ppm→3  ch4_ppb→4
    RC->>RC: build CreateTableRequest<br/>{name, fresh_schema, partition_spec, properties}

    Note over RC,PG: ── Network: POST to REST Catalog ──

    RC->>SRV: POST /v1/namespaces/pyiceberg_demo/tables
    SRV->>S3: Write metadata/00000-uuid.metadata.json<br/>{schema, field IDs, empty snapshots:[]}
    S3-->>SRV: ✅ written
    SRV->>PG: INSERT table record<br/>(namespace, name, metadata_location)
    PG-->>SRV: ✅ done
    SRV-->>RC: 200 OK + TableResponse<br/>{metadata_location, metadata, config}

    Note over RC,SC: ── Client-side: build Table object ──

    RC->>RC: _response_to_table()<br/>Table(identifier, metadata, FileIO → S3)
    RC-->>Code: Table object (empty — no data yet)

    Note over Code,PG: Field IDs 1-4 are now PERMANENT — renaming a column never changes its ID
```

**What just happened:**
- `RestCatalog.create_table()` first calls `assign_fresh_schema_ids(schema)` in `schema.py` — **100% client-side**, no network yet
- Each field gets a **permanent integer ID**: `sensor_id→1`, `ts→2`, `co2_ppm→3`, `ch4_ppb→4`
- Only then does PyIceberg send `POST /v1/namespaces/pyiceberg_demo/tables` to the REST server
- The server writes **exactly one file** to S3: `metadata/00000-<uuid>.metadata.json` — no `data/` folder yet
- The server registers the table in PostgreSQL, then returns the metadata
- **Field IDs are permanent** — if you later rename `co2_ppm` → `co2_concentration`, its ID stays `3`. That's how Iceberg tracks schema evolution safely across millions of files

### Verify empty table metadata

**What we're doing:** Inspecting the freshly created table to confirm it exists but has no data yet.

In [ ]:
import json

meta = table.metadata
print(f"Format version : {meta.format_version}")
print(f"Table UUID     : {meta.table_uuid}")
print(f"Location       : {meta.location}")
print(f"Current snap   : {meta.current_snapshot_id}")
print(f"Snapshots      : {len(meta.snapshots)}")
print(f"\nMetadata file on S3:\n  {table.metadata_location}\n")

with table.io.new_input(table.metadata_location).open() as f:
    print(json.dumps(json.loads(f.read()), indent=2))

**Reading the metadata.json — what each field means:**

```jsonc
{
  "format-version": 2,              ← Iceberg V2 (supports delete files, row-level deletes)
  "table-uuid": "6cf047cd-...",     ← permanent identity — never changes even if renamed
  "location": "s3://warehouse/...", ← root S3 path — all data + metadata lives under here
  "last-column-id": 4,              ← next new column gets ID 5
  "current-schema-id": 0,           ← which entry in "schemas" array is active
  "schemas": [{ "fields": [
      {"id": 1, "name": "sensor_id"},  ← field IDs are PERMANENT — rename won't break old reads
      {"id": 2, "name": "ts"},
      {"id": 3, "name": "co2_ppm"},
      {"id": 4, "name": "ch4_ppb"}
  ]}],
  "partition-specs": [{"fields": []}],  ← empty = unpartitioned
  "sort-orders":     [{"fields": []}],  ← empty = unsorted
  "properties": { "write.parquet.compression-codec": "zstd" },
  "current-snapshot-id": -1,  ← NO DATA YET
  "snapshots": [],             ← empty — first append() adds an entry here
  "metadata-log": []           ← history of previous metadata files
}
```
> After `table.append(batch_1)` — `current-snapshot-id` becomes a real 64-bit integer and `snapshots` gets one entry pointing to the manifest list Avro file on S3.

**Expected output:**
```
format_version : 2
table_uuid     : 6cf047cd-...
location       : s3://warehouse/pyiceberg_demo/sensor_readings
current_snap   : None   ← no data written yet
```
> **Try it:** Open MinIO at [localhost:9001](http://localhost:9001) → `warehouse/pyiceberg_demo/sensor_readings/metadata/` — you should see exactly one file: `00000-<uuid>.metadata.json`

---
## 4. Write Path — Append Data

The write path is the heart of this workshop. When you call `table.append(df)`, PyIceberg goes through 6 stages:

```
Stage 1: Transaction open              (IN MEMORY — nothing written)
Stage 2: Schema validation             (IN MEMORY — PyArrow ↔ Iceberg check)
Stage 3: DataFrame → Parquet → S3      (S3 WRITE #1 — your actual data)
Stage 4: Manifest file → S3            (S3 WRITE #2 — file index with stats)
Stage 5: Manifest list + Snapshot → S3 (S3 WRITE #3 — snapshot TOC)
Stage 6: Catalog commit                (ATOMIC GATE — HTTP POST)
```

**Nothing is visible to readers until Stage 6 completes.**

### Generate sample data

**What we're doing:** Creating a helper function that generates synthetic Picarro sensor data. Each batch = 50 rows with random sensor IDs, timestamps, CO2, and CH4 readings.

In [ ]:
import pyarrow as pa
import datetime
import random

def make_batch(batch_id: int, n_rows: int = 50) -> pa.Table:
    """Generate a batch of synthetic sensor readings."""
    base_ts = datetime.datetime(2024, 1, 1, 0, 0, 0) + datetime.timedelta(days=batch_id)
    return pa.table({
        "sensor_id": [f"PICARRO-{random.randint(1,5):03d}" for _ in range(n_rows)],
        "ts":        [base_ts + datetime.timedelta(minutes=i) for i in range(n_rows)],
        "co2_ppm":   [400.0 + random.gauss(0, 5) for _ in range(n_rows)],
        "ch4_ppb":   [1850.0 + random.gauss(0, 20) for _ in range(n_rows)],
    })

batch = make_batch(1)
print(f"Batch shape: {batch.shape}")
print(f"Schema:\n{batch.schema}")
print()
print(batch.to_pandas().head())

**Expected output:**
```
Batch shape: (50, 4)
Schema:
sensor_id: string
ts: timestamp[us]
co2_ppm: double
ch4_ppb: double
```
Plus a DataFrame preview with 5 rows of sensor data.

> **Note:** `pa.table()` creates a PyArrow Table — PyArrow infers `timestamp[us]` (microsecond precision) and `double` (64-bit float). During `table.append()`, Stage 2 validates these types match the Iceberg schema before any S3 write.

### Append 1 batch — watch the full write path happen once

**What we're doing:** Appending a single batch of 50 rows. One `table.append()` call runs all 6 stages of the write path: schema validation → Parquet write → manifest → manifest list → atomic catalog commit. After this cell: **1 snapshot, 1 Parquet file, 1 manifest, 1 manifest list.**

In [ ]:
total_rows = 0

batch = make_batch(1)
table.append(batch)
total_rows += len(batch)

snap = table.current_snapshot()
print(f"--- Batch 1 appended ({len(batch)} rows) ---")
print(f"  snapshot_id : {snap.snapshot_id}")
print(f"  timestamp   : {snap.timestamp_ms}")
print(f"  operation   : {snap.summary.operation.value}")
print(f"  added-data-files  : {snap.summary.get('added-data-files', 'N/A')}")
print(f"  added-records     : {snap.summary.get('added-records', 'N/A')}")
print(f"  total-records     : {snap.summary.get('total-records', 'N/A')}")
print(f"\nTotal rows written so far: {total_rows}")

**Expected output:**
```
--- Batch 1 appended (50 rows) ---
  snapshot_id       : 2985870393337822537   ← your ID will differ
  timestamp         : 1744099523000
  operation         : append
  added-data-files  : 1
  added-records     : 50
  total-records     : 50

Total rows written so far: 50
```

> **Check MinIO now:** `warehouse/pyiceberg_demo/sensor_readings/`
> - `data/` → **1** `.parquet` file
> - `metadata/` → **1** manifest (`.avro`) + **1** manifest list (`snap-*.avro`) + **2** `metadata.json` files

**Sequence diagram — what just happened inside `table.append(batch)`:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant TX as Transaction<br/><br/>table/__init__.py L209
    participant FA as _FastAppendFiles<br/><br/>table/update/snapshot.py L499
    participant PY as io/pyarrow.py<br/><br/>write_parquet()
    participant S3 as MinIO / S3
    participant MF as manifest.py
    participant CAT as REST Catalog Server

    Code->>TX: table.append(batch)
    TX->>FA: _FastAppendFiles initialized

    Note over TX,PY: Stage 1–2: Validate schema (client-side, no network)
    TX->>PY: _check_pyarrow_schema_compatible()
    PY-->>TX: ✅ Arrow dtypes match Iceberg types

    Note over PY,S3: Stage 3: Write Parquet — S3 write #1
    TX->>PY: bin_pack_arrow_table() → write_parquet()
    PY->>S3: PUT data/00000-0-uuid.parquet
    S3-->>PY: ✅ written
    PY-->>TX: DataFile + column stats (min/max, null counts)

    Note over MF,S3: Stage 4: Write Manifest file (Avro) — S3 write #2
    TX->>MF: ManifestEntry(DataFile)
    MF->>S3: PUT metadata/uuid-m0.avro
    S3-->>MF: ✅ written

    Note over FA,S3: Stage 5: Write Manifest List (Avro) — S3 write #3
    FA->>S3: PUT metadata/snap-id-uuid.avro
    S3-->>FA: ✅ written + Snapshot object created

    Note over TX,CAT: Stage 6: Atomic Catalog Commit — THE GATE
    TX->>CAT: POST /v1/tables/commit<br/>AddSnapshotUpdate + AssertRefSnapshotId
    CAT-->>TX: 200 OK — snapshot is now visible to all readers ✅
    TX-->>Code: return

    Note over TX,CAT: If another writer committed first → 409 Conflict (optimistic lock)
```

**What just happened (each append = 4 files + 1 atomic commit):**
- **Stage 1–2 (client-side):** `Transaction` opens, `_FastAppendFiles` initializes, Arrow dtypes validated against Iceberg schema — *no network yet*
- **Stage 3 — S3 write #1:** `write_parquet()` uploads the Parquet file and computes per-column `min/max/null` stats stored in the `DataFile` object
- **Stage 4 — S3 write #2:** `ManifestEntry` wraps the `DataFile` → written as Avro (`.avro`). This is how Iceberg indexes file-level stats for fast pruning
- **Stage 5 — S3 write #3:** `write_manifest_list()` writes the manifest list (another Avro file) pointing to all manifests, and creates a `Snapshot`
- **Stage 6 — Atomic commit:** `catalog.commit_table()` sends an HTTP POST with `AddSnapshotUpdate + AssertRefSnapshotId`. The `AssertRefSnapshotId` is the **optimistic lock** — if two writers race, only one wins; the other gets a `409 Conflict` and retries
- **Zero partial reads:** readers either see the old snapshot or the new one — never half-written data

### Now append 2 more batches — build up the snapshot chain

**What we're doing:** Appending batches 2 and 3 so we have 3 snapshots total. The Inspect section (Section 5) will walk this 3-snapshot chain. Each append is identical to what you just saw — same 6 stages, new snapshot, old ones untouched.

In [ ]:
for b in range(2, 4):
    batch = make_batch(b)
    table.append(batch)
    total_rows += len(batch)
    snap = table.current_snapshot()
    print(f"--- Batch {b} appended ({len(batch)} rows) | total-records: {snap.summary.get('total-records', 'N/A')} ---")

print(f"\nTotal rows written: {total_rows}")
print(f"Snapshots: {len(table.history())}")

**Expected output:**
```
--- Batch 2 appended (50 rows) | total-records: 100 ---
--- Batch 3 appended (50 rows) | total-records: 150 ---

Total rows written: 150
Snapshots: 3
```

> **Check MinIO:** `data/` → 3 `.parquet` files | `metadata/` → 3 manifests + 3 manifest lists + 4 `metadata.json` files

---
## 4b. Write Path — Partitioned Table

**What we're doing:** Creating a second table with a partition spec (`sensor_id`) and appending data. Partitioning changes the write path: Iceberg groups rows by partition value before writing, and the metadata layer records partition boundaries — enabling the read path to skip entire files without even opening them.

> **Key insight:** The partition spec lives in the table metadata — not in the Parquet files themselves. Iceberg "knows" where things are without touching a single data file.

In [ ]:
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform

# Partition by sensor_id (identity transform — one partition per distinct value)
partition_spec = PartitionSpec(
    PartitionField(
        source_id=1,           # field ID 1 = sensor_id
        field_id=1000,
        transform=IdentityTransform(),
        name="sensor_id",
    )
)

PARTITIONED_TABLE_ID = (*NS, "sensor_readings_partitioned")

# Drop and recreate for clean runs
try:
    catalog.drop_table(PARTITIONED_TABLE_ID)
    print("(dropped existing partitioned table)")
except Exception:
    pass

part_table = catalog.create_table(
    identifier=PARTITIONED_TABLE_ID,
    schema=schema,
    partition_spec=partition_spec,
)
print(f"Partitioned table created: {PARTITIONED_TABLE_ID[-1]}")
print(f"Partition spec: {part_table.metadata.partition_specs[0]}")

**Expected output:**
```
(dropped existing partitioned table)    ← only if re-running
Partitioned table created: sensor_readings_partitioned
Partition spec: PartitionSpec(fields=[PartitionField(source_id=1, field_id=1000, transform=IdentityTransform(), name='sensor_id')])
```

### Append data to the partitioned table

**What we're doing:** Appending 150 rows (3 batches × 50 rows, each batch has 5 sensor IDs). Iceberg will write **one Parquet file per partition value** — so 5 files (one per sensor_id: PICARRO-001 through PICARRO-005).

In [ ]:
import pyarrow as pa

# Build a single batch covering all 5 sensor_ids so each gets its own partition
all_batches = pa.concat_tables([make_batch(b) for b in range(1, 4)])
part_table.append(all_batches)

snap = part_table.current_snapshot()
print(f"Snapshot ID   : {snap.snapshot_id}")
print(f"Added files   : {snap.summary.get('added-data-files', 'N/A')}")
print(f"Total records : {snap.summary.get('total-records', 'N/A')}")

# Show partition paths in MinIO
print("\nData files written:")
for entry in part_table.inspect.files().to_pydict()["file_path"]:
    # show just the partition folder + filename
    parts = entry.split("/")
    print(f"  .../{'/'.join(parts[-3:])}")

**Expected output:**
```
Snapshot ID   : 7483920183...
Added files   : 5
Total records : 150

Data files written:
  .../sensor_id=PICARRO-001/00000-0-<uuid>.parquet
  .../sensor_id=PICARRO-002/00000-0-<uuid>.parquet
  .../sensor_id=PICARRO-003/00000-0-<uuid>.parquet
  .../sensor_id=PICARRO-004/00000-0-<uuid>.parquet
  .../sensor_id=PICARRO-005/00000-0-<uuid>.parquet
```

> **Compare to the unpartitioned table:** That had 3 files (1 per append). This partitioned table has 5 files (1 per sensor_id value) — stored under `sensor_id=PICARRO-001/` folders in S3.

**Sequence diagram — partitioned write path:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant TX as Transaction<br/><br/>table/__init__.py
    participant FA as _FastAppendFiles<br/><br/>table/update/snapshot.py L499
    participant PY as io/pyarrow.py<br/><br/>write_parquet()
    participant S3 as MinIO / S3
    participant MF as manifest.py
    participant CAT as REST Catalog Server

    Code->>TX: part_table.append(all_batches)
    TX->>FA: _FastAppendFiles initialized

    Note over TX,PY: Extra step vs. unpartitioned: bin rows by partition value
    TX->>PY: bin_pack_arrow_table()<br/>groups rows by sensor_id → 5 WriteTask(s)

    loop for each partition value (PICARRO-001 … PICARRO-005)
        PY->>S3: PUT data/sensor_id=PICARRO-N/00000-0-uuid.parquet
        S3-->>PY: ✅
    end

    PY-->>TX: 5 × DataFile with partition_value + column stats

    TX->>MF: ManifestEntry for each DataFile<br/>records partition_value in manifest
    MF->>S3: PUT metadata/uuid-m0.avro  (1 manifest, 5 entries)
    S3-->>MF: ✅

    Note over FA,S3: Stage 5: Write Manifest List — S3 write #3
    FA->>S3: PUT metadata/snap-id-uuid.avro
    S3-->>FA: ✅ Snapshot created

    Note over TX,CAT: Stage 6: Atomic Catalog Commit
    TX->>CAT: POST /v1/tables/commit  (AddSnapshotUpdate + AssertRefSnapshotId)
    CAT-->>TX: 200 OK ✅
    TX-->>Code: return
```

**What just happened:**
- `bin_pack_arrow_table()` groups rows by partition value before writing — **5 groups → 5 Parquet files**, one per `sensor_id`
- Each `DataFile` in the manifest records its `partition_value` (e.g. `sensor_id=PICARRO-001`) alongside column stats
- The manifest file has 5 entries (one per file) vs. 1 entry in the unpartitioned case
- **Read-time benefit:** a query `WHERE sensor_id = 'PICARRO-003'` uses `_ManifestEvalVisitor` to check each file's recorded `partition_value` — it opens **1 file** instead of all 5, without reading any Parquet
- Partition folders (`sensor_id=PICARRO-001/`) in S3 are a side effect of the path format — the actual partition metadata lives in the **manifest**, not the folder name

---
## 4c. Write Path — Overwrite

**What we're doing:** Using `part_table.overwrite()` to replace all rows in the `PICARRO-001` partition with fresh data. We use the **partitioned table** from 4b because the overwrite can then cleanly target exactly one partition file — ADDED for the new file, DELETED for the old one, zero other files touched.

> **Append vs. Overwrite:** `append()` only adds new files. `overwrite()` marks old files as `DELETED` in the manifest and writes a new file — both committed atomically. No in-place mutation, ever.

In [ ]:
from pyiceberg.expressions import EqualTo
import pyarrow.compute as pc

# Check state of PARTITIONED table before overwrite
before_snap = part_table.current_snapshot()
before_df = part_table.scan().to_arrow()
before_count = before_df.num_rows
p001_count = pc.sum(pc.equal(before_df["sensor_id"], "PICARRO-001")).as_py()
print(f"Before overwrite : {before_count} total rows")
print(f"  PICARRO-001 rows in current snapshot: {p001_count}")

# Build EXPLICIT replacement rows — all PICARRO-001, fresh timestamps & elevated co2
replacement = pa.table({
    "sensor_id": ["PICARRO-001"] * 20,
    "ts":        [datetime.datetime(2024, 6, 1) + datetime.timedelta(minutes=i) for i in range(20)],
    "co2_ppm":   [420.0 + i * 0.5 for i in range(20)],   # clearly new values (was ~400)
    "ch4_ppb":   [1900.0 + float(i) for i in range(20)],
})
print(f"\nReplacement rows : {len(replacement)}  (all PICARRO-001, elevated co2_ppm ~420+)")

# Overwrite the PICARRO-001 partition with replacement data
part_table.overwrite(
    df=replacement,
    overwrite_filter=EqualTo("sensor_id", "PICARRO-001"),
)

after_snap = part_table.current_snapshot()
after_count = part_table.scan().to_arrow().num_rows
print(f"\nAfter overwrite  : {after_count} total rows  (was {before_count})")
print(f"  PICARRO-001 rows now: 20  (was {p001_count})")
print(f"Operation        : {after_snap.summary.operation.value}")
print(f"Added files      : {after_snap.summary.get('added-data-files', 'N/A')}")
print(f"Deleted files    : {after_snap.summary.get('deleted-data-files', 'N/A')}")

**Expected output:**
```
Before overwrite : 150 total rows
  PICARRO-001 rows in current snapshot: ~30   ← varies (random data from 4b)

Replacement rows : 20  (all PICARRO-001, elevated co2_ppm ~420+)

After overwrite  : ~140 total rows  (was 150)   ← 30 removed, 20 added
  PICARRO-001 rows now: 20  (was ~30)
Operation        : overwrite
Added files      : 1    ← new PICARRO-001 partition file
Deleted files    : 1    ← old PICARRO-001 partition file
```

> **Key point:** Only the `sensor_id=PICARRO-001` partition file was touched. The other 4 partition files (`PICARRO-002` through `PICARRO-005`) were **never opened** — the manifest tells Iceberg exactly which file to replace.

**Sequence diagram — overwrite vs. append:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant TX as Transaction<br/><br/>table/__init__.py
    participant PY as io/pyarrow.py
    participant S3 as MinIO / S3
    participant MF as manifest.py
    participant CAT as REST Catalog Server

    Code->>TX: table.overwrite(new_data, overwrite_filter=EqualTo("sensor_id","PICARRO-001"))

    Note over TX,PY: Extra step vs. append: scan to find files matching the filter
    TX->>S3: scan current snapshot → find files containing PICARRO-001 rows
    S3-->>TX: [file1.parquet, file3.parquet]  (the affected files)

    Note over PY,S3: Write new Parquet file (same as append Stage 3)
    TX->>PY: write_parquet(new_data)
    PY->>S3: PUT data/00000-0-uuid-new.parquet
    S3-->>PY: ✅ DataFile returned

    Note over MF,S3: Manifest records BOTH new file AND deleted files
    TX->>MF: ManifestEntry(new DataFile, status=ADDED)<br/>ManifestEntry(old DataFile, status=DELETED)
    MF->>S3: PUT metadata/uuid-m0.avro
    S3-->>MF: ✅

    TX->>S3: PUT metadata/snap-id-uuid.avro  (manifest list)
    S3-->>TX: ✅

    Note over TX,CAT: Atomic commit — same gate as append
    TX->>CAT: POST /v1/tables/commit  (overwrite + AssertRefSnapshotId)
    CAT-->>TX: 200 OK ✅
    TX-->>Code: return (old files still in S3, just dereferenced)
```

**What just happened:**
- Overwrite first **scans the current snapshot** to find which files contain rows matching `sensor_id = 'PICARRO-001'` — this is extra work vs. a pure append
- A new Parquet file is written with the replacement rows (same Stage 3 as append)
- The manifest records two kinds of entries: `ADDED` (new file) and `DELETED` (old files that had PICARRO-001 rows)
- The atomic commit is identical to append — `AssertRefSnapshotId` ensures no other writer snuck in
- **Old files stay on S3** — they are only dereferenced from the new snapshot. Readers on the old snapshot see the original data (time travel still works). Run `table.expire_snapshots()` to eventually clean up unreferenced files
- **This is how Iceberg achieves ACID for updates** — no in-place mutation, just a new snapshot that points to a new file layout

---
## 5. Inspect: Snapshots, Manifests & Data Files

**What we're doing:** Peeling back the metadata layers of the `sensor_readings` table. Iceberg stores everything in three levels — snapshot history → manifest list → manifest files → data files. We'll walk each level and see what's actually on S3.

> **Key insight:** `table.inspect.*` gives you a live PyArrow table view of Iceberg's metadata. No Spark, no JVM — pure Python.

### 5a. Snapshot History

In [ ]:
import pyarrow as pa

# ── Snapshot history ─────────────────────────────────────────────────────────
snapshots_df = table.inspect.snapshots()
print("=== Snapshot History ===")
print(snapshots_df.to_pandas()[
    ["committed_at", "snapshot_id", "parent_id", "operation", "summary"]
].to_string(index=False))

print(f"\nTotal snapshots: {len(table.history())}")
for entry in table.history():
    print(f"  snapshot_id={entry.snapshot_id}  timestamp_ms={entry.timestamp_ms}")

**Expected output:**
```
=== Snapshot History ===
 committed_at   snapshot_id   parent_id  operation                        summary
 ...            2985870...    None       append     {'added-data-files': '1', ...}
 ...            7483920...    2985870... append     {'added-data-files': '1', ...}
 ...            1029384...    7483920... append     {'added-data-files': '1', ...}

Total snapshots: 3
```

- Each append created a new snapshot; `parent_id` chains them together
- The `summary` dict is what you see in `snap.summary` — it comes straight from the metadata JSON

### 5b. Manifest Files